In [2]:
import pandas as pd
import numpy as np

In [3]:
import os

from io import BytesIO
from datetime import datetime, timezone
import boto3
from botocore.client import Config
from dotenv import load_dotenv


load_dotenv()

BUCKET_NAME = os.environ["MINIO_BUCKET"]

s3_client = boto3.client(
    "s3",
    endpoint_url=os.environ["MINIO_ENDPOINT"],
    aws_access_key_id=os.environ["MINIO_ACCESS_KEY"],
    aws_secret_access_key=os.environ["MINIO_SECRET_KEY"],
    region_name="us-east-1",
    config=Config(signature_version="s3v4"),
)

In [4]:
def read_parquet_prefix(s3_client, bucket_name, prefix):
    """
    Read every Parquet object under a MinIO prefix
    and return one pandas DataFrame.
    """
    #MinIO may contain many objects. Instead of returning all objects in one response, the S3 API may divide them into pages.
    paginator = s3_client.get_paginator("list_objects_v2")

    #This list will store the names of all Parquet objects found.
    parquet_keys = []

    #This asks MinIO: Show me all objects inside this bucket whose key starts with this prefix.
    for page in paginator.paginate(Bucket=bucket_name,Prefix=prefix):
        #Contents contains the objects found by MinIO.
        for obj in page.get("Contents", []):
            #The key is the complete MinIO path of the file.
            key = obj["Key"]

            if key.endswith(".parquet"):
                parquet_keys.append(key)

    if not parquet_keys:
        raise FileNotFoundError(
            f"No Parquet files found under prefix: {prefix}"
        )

    dataframes = []

    for key in parquet_keys:
        response = s3_client.get_object(
            Bucket=bucket_name,
            Key=key
        )

        parquet_bytes = response["Body"].read()

        dataframe = pd.read_parquet(
            BytesIO(parquet_bytes),
            engine="fastparquet"
        )

        dataframes.append(dataframe)

        print(f"Loaded: {key}")
        print(f"Shape: {dataframe.shape}")

    combined_dataframe = pd.concat(
        dataframes,
        ignore_index=True
    )

    return combined_dataframe

def read_parquet_object(s3_client, bucket_name, object_key):
    response = s3_client.get_object(Bucket=bucket_name, Key=object_key)
    return pd.read_parquet(
        BytesIO(response["Body"].read()),
        engine="fastparquet",
    )

In [5]:
metadata_df = read_parquet_prefix(
    s3_client,
    BUCKET_NAME,
    "metadata/etl_metadata/"
)

required_raw_datasets = {"train_data", "test_data", "RUL_data"}
successful_raw_metadata = metadata_df[
    (metadata_df["layer"] == "raw")
    & (metadata_df["status"] == "success")
    & metadata_df["dataset_name"].isin(required_raw_datasets)
]

datasets_by_run = successful_raw_metadata.groupby("run_id")["dataset_name"].apply(set)
valid_raw_run_ids = sorted(
    run_id
    for run_id, datasets in datasets_by_run.items()
    if required_raw_datasets.issubset(datasets)
)

if not valid_raw_run_ids:
    raise FileNotFoundError("No successful Raw run contains train, test, and RUL data.")

pipeline_run_id = valid_raw_run_ids[-1]
selected_raw_metadata = successful_raw_metadata[
    successful_raw_metadata["run_id"] == pipeline_run_id
]
pipeline_load_date = selected_raw_metadata["load_date"].iloc[0]
metadata_key = (
    f"metadata/etl_metadata/load_date={pipeline_load_date}/"
    f"run_id={pipeline_run_id}/manifest.parquet"
)

def raw_object_key(dataset_name):
    matches = selected_raw_metadata[
        selected_raw_metadata["dataset_name"] == dataset_name
    ]
    if len(matches) != 1:
        raise ValueError(f"Expected one Raw object for {dataset_name}.")
    return matches.iloc[0]["object_key"]

train_df_minio = read_parquet_object(
    s3_client, BUCKET_NAME, raw_object_key("train_data")
)
test_df_minio = read_parquet_object(
    s3_client, BUCKET_NAME, raw_object_key("test_data")
)
rul_df_minio = read_parquet_object(
    s3_client, BUCKET_NAME, raw_object_key("RUL_data")
)

print(f"Selected Raw run: {pipeline_run_id}")

Loaded: metadata/etl_metadata/load_date=2026-09-02/run_id=20260902T120041719003Z/manifest.parquet
Shape: (9, 8)
Selected Raw run: 20260902T120041719003Z


In [6]:
train_df_minio.columns = train_df_minio.columns.str.lower()
test_df_minio.columns = test_df_minio.columns.str.lower()
rul_df_minio.columns = rul_df_minio.columns.str.lower()

## Silver validation checks

In [7]:
# Create Silver working copies so the Raw DataFrames remain unchanged.
silver_train_df = train_df_minio.copy()
silver_test_df = test_df_minio.copy()
silver_rul_df = rul_df_minio.copy()

raw_df_train = silver_train_df
raw_df_test = silver_test_df
raw_df_RUL = silver_rul_df

In [8]:
columns = [
    "unit_number" if i == 0
    else "time_in_cycles" if i == 1
    else f"operational_setting_{i-1}" if i <= 4
    else f"sensor_measurement_{i-4}"
    for i in range(26)
]

In [14]:
columns != list(raw_df_train.columns)

False

In [15]:
if len(raw_df_train.columns) != len(columns):
    raise ValueError("Number of columns in train df is wrong")
if len(raw_df_test.columns) != len(columns):
    raise ValueError("Number of columns in test df is wrong")

if columns != list(raw_df_train.columns):
    raise ValueError("train df contains wrong columns")
if columns != list(raw_df_test.columns):
    raise ValueError("test df contains wrong columns")


In [18]:
dataframes = {
    "train": raw_df_train,
    "test": raw_df_test,
    "RUL": raw_df_RUL,
}

for dataset_name, dataframe in dataframes.items():
    null_counts = dataframe.isna().sum()
    columns_with_nulls = null_counts[null_counts > 0]

    if not columns_with_nulls.empty:
        raise ValueError(
            f"{dataset_name} contains null values: "
            f"{columns_with_nulls.to_dict()}"
        )

In [16]:
expected_rul_columns = [
    "unit_number",
    "remaining_useful_life",
]

if list(raw_df_RUL.columns) != expected_rul_columns:
    raise ValueError(
        f"RUL dataframe has the wrong schema. "
        f"Expected {expected_rul_columns}, "
        f"received {list(raw_df_RUL.columns)}"
    )

In [20]:
#data contains no duplicates
new = raw_df_train.drop_duplicates()
print("Number of duplicated rows in train df : ",len(raw_df_train)-len(new))
print("Number of duplicated rows in train df (based only on unit_number and cycles) :",raw_df_train.duplicated(subset=["unit_number", "time_in_cycles"]).sum())
raw_df_train = raw_df_train.drop_duplicates().reset_index(drop=True)
if raw_df_train.duplicated(subset=["unit_number", "time_in_cycles"], keep=False).any():
    raise ValueError("Train data contains conflicting records for the same engine and cycle.")
silver_train_df = raw_df_train

new = raw_df_test.drop_duplicates()
print("\nNumber of duplicated rows in test df : ",len(raw_df_test)-len(new))
print("Number of duplicated rows in test df (based only on unit_number and cycles) :",raw_df_test.duplicated(subset=["unit_number", "time_in_cycles"]).sum())
raw_df_test = raw_df_test.drop_duplicates().reset_index(drop=True)
if raw_df_test.duplicated(subset=["unit_number", "time_in_cycles"], keep=False).any():
    raise ValueError("Test data contains conflicting records for the same engine and cycle.")
silver_test_df = raw_df_test

new = raw_df_RUL.drop_duplicates()
print("\nNumber of duplicated rows : ",len(raw_df_RUL)-len(new))
print("Number of duplicated rows (based only on unit_number) :",raw_df_RUL.duplicated(subset=["unit_number"]).sum())
raw_df_RUL = raw_df_RUL.drop_duplicates().reset_index(drop=True)
silver_rul_df = raw_df_RUL

Number of duplicated rows in train df :  0
Number of duplicated rows in train df (based only on unit_number and cycles) : 0

Number of duplicated rows in test df :  0
Number of duplicated rows in test df (based only on unit_number and cycles) : 0

Number of duplicated rows :  0
Number of duplicated rows (based only on unit_number) : 0


In [19]:
from pandas.api.types import is_integer_dtype

for dataset_name, dataframe in dataframes.items():
    if not is_integer_dtype(dataframe["unit_number"]):
        raise TypeError(
            f"{dataset_name}.unit_number must have an integer dtype, "
            f"but received {dataframe['unit_number'].dtype}"
        )

    if (dataframe["unit_number"] <= 0).any():
        invalid_ids = (
            dataframe.loc[
                dataframe["unit_number"] <= 0,
                "unit_number",
            ]
            .unique()
            .tolist()
        )

        raise ValueError(
            f"{dataset_name} contains non-positive engine IDs: "
            f"{invalid_ids[:10]}"
        )

In [21]:
for dataset_name, dataframe in {
    "train": raw_df_train,
    "test": raw_df_test,
}.items():
    if not is_integer_dtype(dataframe["time_in_cycles"]):
        raise TypeError(
            f"{dataset_name}.time_in_cycles must have an integer dtype"
        )

    if (dataframe["time_in_cycles"] <= 0).any():
        raise ValueError(
            f"{dataset_name} contains non-positive cycle numbers"
        )

In [ ]:
for dataset_name, dataframe in dataframes.items():
    numeric_dataframe = dataframe.select_dtypes(include=np.number)

    finite_mask = np.isfinite(
        numeric_dataframe.to_numpy(dtype=float)
    )

    if not finite_mask.all():
        bad_rows, bad_columns = np.where(~finite_mask)

        examples = [
            {
                "row": int(numeric_dataframe.index[row]),
                "column": numeric_dataframe.columns[column],
                "value": numeric_dataframe.iloc[row, column],
            }
            for row, column in zip(
                bad_rows[:10],
                bad_columns[:10],
            )
        ]

        raise ValueError(
            f"{dataset_name} contains non-finite numeric values. "
            f"Examples: {examples}"
        )

In [10]:
test_engines = raw_df_test["unit_number"].unique()
print("Number of test engines :",len(test_engines))
print("Number of rows in test df :",len(raw_df_test))
print("Number of columns in test df :",raw_df_test.shape[1])

Number of test engines : 100
Number of rows in test df : 13096
Number of columns in test df : 26


In [11]:
raw_df_test = raw_df_test.sort_values(
    by=["unit_number", "time_in_cycles"]
).reset_index(drop=True)
test_engines = raw_df_test["unit_number"].unique()
silver_test_df = raw_df_test

In [ ]:
test = raw_df_test.groupby('unit_number').cumcount() + 1
copy_df = raw_df_test.copy()
copy_df["row_number"] = test
if len(copy_df[copy_df["time_in_cycles"]!=copy_df["row_number"]])>0:
    raise ValueError(f"Missing time sequence in test df")

In [ ]:
raw_df_train = raw_df_train.sort_values(
    by=["unit_number", "time_in_cycles"]
).reset_index(drop=True)
silver_train_df = raw_df_train
test = raw_df_train.groupby('unit_number').cumcount() + 1
copy_df = raw_df_train.copy()
copy_df["row_number"] = test
if len(copy_df[copy_df["time_in_cycles"]!=copy_df["row_number"]])>0:
    raise ValueError(f"Missing time sequence in train df")

In [14]:
temp = raw_df_RUL["unit_number"].value_counts()
if len(temp[temp>1])!=0:
    raise ValueError(f"RUL df contains multiple rows per engine")

In [15]:
if len(raw_df_RUL) != len(test_engines):
    raise ValueError("The number of RUL values does not match the number of test engines.")

if raw_df_RUL["remaining_useful_life"].isna().any():
    raise ValueError("RUL contains missing values.")

In [16]:
if list(test_engines) != list(raw_df_RUL["unit_number"]):
    raise ValueError(
        "The engine order in the RUL file does not match the test data."
    )

In [17]:
if len(raw_df_RUL[raw_df_RUL["remaining_useful_life"]<0]) > 0:
    raise ValueError(f"Existence of negative value in remaining_useful_life")

In [18]:
## Silver layer

#The Silver DataFrames are created from the DataFrames read from MinIO Raw. The original Raw DataFrames are not modified.

In [19]:
from datetime import datetime, timezone

silver_processing_timestamp = datetime.now(timezone.utc).isoformat()
silver_data_quality_status = "passed_silver_validation"

# The Silver DataFrames were cleaned and validated above.
for dataframe in (silver_train_df, silver_test_df, silver_rul_df):
    dataframe["processing_timestamp"] = silver_processing_timestamp
    dataframe["data_quality_status"] = silver_data_quality_status

print(f"Silver train shape: {silver_train_df.shape}")
print(f"Silver test shape: {silver_test_df.shape}")
print(f"Silver RUL shape: {silver_rul_df.shape}")

Silver train shape: (20631, 28)
Silver test shape: (13096, 28)
Silver RUL shape: (100, 4)


The following cell writes the cleaned and validated Silver DataFrames to separate Silver prefixes. The added metadata records when Silver processing occurred and the Silver validation result.

In [20]:
silver_dataframes = {
    "train_data": silver_train_df,
    "test_data": silver_test_df,
    "RUL_data": silver_rul_df,
}
silver_metadata_records = []

for dataset_name, dataframe in silver_dataframes.items():
    parquet_buffer = BytesIO()

    dataframe.to_parquet(
        parquet_buffer,
        index=False,
        engine="fastparquet",
    )

    parquet_buffer.seek(0)
    object_key = (
        f"silver/{dataset_name}/"
        f"load_date={pipeline_load_date}/"
        f"run_id={pipeline_run_id}/"
        f"{dataset_name}.parquet"
    )

    s3_client.put_object(
        Bucket=BUCKET_NAME,
        Key=object_key,
        Body=parquet_buffer.getvalue(),
        ContentType="application/octet-stream",
    )

    print(f"Uploaded Silver dataset to MinIO: {object_key}")
    silver_metadata_records.append({
        "run_id": pipeline_run_id,
        "load_date": pipeline_load_date,
        "layer": "silver",
        "dataset_name": dataset_name,
        "object_key": object_key,
        "status": "success",
        "row_count": len(dataframe),
        "recorded_at": datetime.now(timezone.utc).isoformat(),
    })

etl_metadata_df = read_parquet_object(
    s3_client, BUCKET_NAME, metadata_key
)
etl_metadata_df = etl_metadata_df[
    ~((etl_metadata_df["run_id"] == pipeline_run_id)
      & (etl_metadata_df["layer"] == "silver"))
]
etl_metadata_df = pd.concat(
    [etl_metadata_df, pd.DataFrame(silver_metadata_records)],
    ignore_index=True,
)
metadata_buffer = BytesIO()
etl_metadata_df.to_parquet(
    metadata_buffer,
    index=False,
    engine="fastparquet",
)
metadata_buffer.seek(0)
s3_client.put_object(
    Bucket=BUCKET_NAME,
    Key=metadata_key,
    Body=metadata_buffer.getvalue(),
    ContentType="application/octet-stream",
)
print(f"Updated ETL metadata manifest: {metadata_key}")

Uploaded Silver dataset to MinIO: silver/train_data/load_date=2026-09-02/run_id=20260902T120041719003Z/train_data.parquet
Uploaded Silver dataset to MinIO: silver/test_data/load_date=2026-09-02/run_id=20260902T120041719003Z/test_data.parquet
Uploaded Silver dataset to MinIO: silver/RUL_data/load_date=2026-09-02/run_id=20260902T120041719003Z/RUL_data.parquet
Updated ETL metadata manifest: metadata/etl_metadata/load_date=2026-09-02/run_id=20260902T120041719003Z/manifest.parquet
